<div style="background:#123B63;color:white;padding:14px 18px;border-radius:6px">
<b>CSE 816 &mdash; Machine Learning Lab</b> &nbsp;&middot;&nbsp; Department of CSE, University of Chittagong<br>
<span style="font-size:90%">Module 2 &middot; Week 4 &middot; Part 1 of 4 &nbsp;&middot;&nbsp; 60 minutes</span>
</div>

# Neurons and Layers

Week 3 ended with a logistic unit: $g(\mathbf{w}\cdot\mathbf{x}+b)$. This week that unit becomes
a **neuron**, and a row of neurons becomes a **layer**. Nothing in the arithmetic is new. What is
new is the bookkeeping &mdash; which weights belong to which unit, which shape every array has
&mdash; and the library, TensorFlow, that does that bookkeeping for you.

This part builds single neurons and single layers in TensorFlow, sets their weights by hand, and
checks every output against NumPy and against the numbers worked on the board. Part 2 trains a
network; Part 3 rebuilds one from scratch; Part 4 is the practice lab.

**Companion theory lecture:** CSE 815, Week 4, Part 1 (Neural Networks) and Part 2 (Neural
Networks in Code).

## What you will be able to do

1. State and check the shape conventions TensorFlow expects: one row per example, one column per
   feature.
2. Move data between NumPy arrays and TensorFlow tensors.
3. Show that a `Dense` neuron with a linear activation *is* linear regression, and one with a
   sigmoid activation *is* logistic regression.
4. Read and set a layer's weights, and say what each axis of the weight matrix means.
5. Assemble a two-layer network with `Sequential` and reproduce the lecture's forward
   propagation exactly.
6. Count the parameters of any fully connected network, by formula and by TensorFlow.

---

## 1. Setup

**TensorFlow needs Python 3.10&ndash;3.13.** If `import tensorflow` fails, check
`python --version` first: there is no stable TensorFlow release for Python 3.14 yet. The
`requirements.txt` file describes how to set up a suitable environment.

TensorFlow prints a few lines of start-up messages; the first line of the cell below hides most
of them. Any that remain are not errors.

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")   # hide TensorFlow's start-up chatter

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential, Input
from tensorflow.keras.layers import Dense

%matplotlib inline
try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")

np.set_printoptions(precision=4, suppress=True)
keras.utils.set_random_seed(816)      # seeds Python, NumPy and TensorFlow together

BLUE, TEAL, AMBER, ROSE, GRAY = "#123B63", "#1B7F79", "#C97B17", "#9B2C4B", "#4A5560"

print("NumPy", np.__version__, "  TensorFlow", tf.__version__)

---

## 2. Data in TensorFlow

Weeks 1&ndash;3 used 1-D arrays freely: `x = np.array([1.0, 2.0])` was a feature vector, and
`X @ w` did the right thing. TensorFlow is stricter. Its layers expect a **2-D matrix with one row
per example and one column per feature**, shape `(m, n)`. A single example is therefore a
$1 \times n$ matrix, not a vector.

In [ ]:
x_1d = np.array([200.0, 17.0])          # 1-D: no rows, no columns
x_row = np.array([[200.0, 17.0]])       # one example, two features
x_col = np.array([[200.0], [17.0]])     # two examples, one feature each

for name, arr in [("x_1d", x_1d), ("x_row", x_row), ("x_col", x_col)]:
    print(f"{name:6} shape {str(arr.shape):8} ndim {arr.ndim}")

assert x_1d.shape == (2,) and x_row.shape == (1, 2) and x_col.shape == (2, 1)

# Converting between the forms is a reshape, not a copy of new data.
assert np.array_equal(x_1d.reshape(1, -1), x_row)
assert np.array_equal(x_1d.reshape(-1, 1), x_col)
print("\nreshape(1, -1) turns one feature vector into one example; reshape(-1, 1) turns it")
print("into a column -- which TensorFlow reads as two examples of one feature each.")

### Tensors

A layer accepts NumPy arrays, but it returns a `tf.Tensor`: TensorFlow's own array type, which
can live on a GPU and records how it was computed so that it can be differentiated during
training. The data are the same; only the container differs. `.numpy()` converts back.

In [ ]:
t = tf.constant(x_row)
print(t)
print("type:", type(t).__name__, "  shape:", t.shape, "  dtype:", t.dtype)

back = t.numpy()
print("\nback to NumPy:", type(back).__name__, back.shape, back.dtype)
assert isinstance(back, np.ndarray) and np.array_equal(back, x_row)

# Note the dtype. tf.constant kept NumPy's float64, but Dense layers compute in float32,
# which carries about 7 significant digits. Comparisons between the two need a tolerance.
print("\na Dense layer's output dtype:", Dense(units=1)(x_row).dtype)
print("float32 machine epsilon:", np.finfo(np.float32).eps)

> **Tolerance.** Every comparison in this lab between a TensorFlow output and a NumPy
> computation uses `np.allclose(..., atol=1e-6)` or looser. Expecting bit-for-bit equality
> between float32 and float64 arithmetic is a bug in the test, not in the model.

---

## 3. A neuron with no activation is linear regression

A `Dense` layer with one unit and `activation="linear"` computes $f(\mathbf{x}) =
\mathbf{w}\cdot\mathbf{x} + b$ and nothing else. Give it the Week 1 flats data and the parameters
gradient descent found there, and it must reproduce Week 1's predictions.

In [ ]:
# Week 1: x = size in 1000 sq ft, y = price in lakh BDT, and the converged fit.
x_train = np.array([0.8, 1.0, 1.2, 1.4, 1.6, 1.8, 2.0, 2.4]).reshape(-1, 1)   # (8, 1)
y_train = np.array([30., 36., 45., 48., 58., 62., 70., 82.])
w_week1, b_week1 = 32.644110, 4.092732

linear_layer = Dense(units=1, activation="linear")

# A layer has no weights until it sees its first input -- only then is n_in known.
print("weights before first call:", linear_layer.get_weights())
_ = linear_layer(x_train[:1])
w0, b0 = linear_layer.get_weights()
print(f"after first call: W shape {w0.shape}, b shape {b0.shape}  (random W, zero b)")

In [ ]:
# set_weights takes a list [W, b] with exactly the shapes get_weights returned.
linear_layer.set_weights([np.array([[w_week1]]), np.array([b_week1])])

f_tf = linear_layer(x_train).numpy()          # shape (8, 1)
f_np = x_train @ np.array([w_week1]) + b_week1   # shape (8,)

print(f"{'size':>6} {'price':>7} {'TensorFlow':>11} {'NumPy':>9}")
print("-" * 36)
for xi, yi, a, b in zip(x_train[:, 0], y_train, f_tf[:, 0], f_np):
    print(f"{xi:6.1f} {yi:7.1f} {a:11.4f} {b:9.4f}")

assert f_tf.shape == (8, 1)
assert np.allclose(f_tf[:, 0], f_np, atol=1e-4)
print("\nThe linear neuron and Week 1's model agree to float32 precision.")

Note the output shape: `(8, 1)`, not `(8,)`. One row per example, one column per unit &mdash; even
when there is only one unit. Use `[:, 0]` or `.ravel()` when you need a flat vector.

---

## 4. A sigmoid neuron is logistic regression

Swap the activation for `"sigmoid"` and the same neuron computes
$g(\mathbf{w}\cdot\mathbf{x}+b)$. Week 3, Part 1 fitted the eight-patient screening set and
found $w = 0.594084$, $b = -2.673380$. Load those parameters into a neuron.

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


# Week 3's screening set: tumour diameter (cm), 1 = malignant.
X_scr = np.arange(1.0, 9.0).reshape(-1, 1)
y_scr = np.array([0., 1, 0, 0, 0, 1, 1, 1])
w_scr, b_scr = 0.594084, -2.673380

logistic_neuron = Sequential([
    Input(shape=(1,)),
    Dense(units=1, activation="sigmoid", name="neuron"),
])
logistic_neuron.get_layer("neuron").set_weights([np.array([[w_scr]]), np.array([b_scr])])

p_tf = logistic_neuron.predict(X_scr, verbose=0)[:, 0]
p_np = sigmoid(X_scr[:, 0] * w_scr + b_scr)

print(f"{'x (cm)':>7} {'y':>3} {'TensorFlow':>11} {'NumPy':>9}")
print("-" * 34)
for xi, yi, a, b in zip(X_scr[:, 0], y_scr, p_tf, p_np):
    print(f"{xi:7.0f} {yi:3.0f} {a:11.4f} {b:9.4f}")

assert np.allclose(p_tf, p_np, atol=1e-6)
assert np.isclose(p_np[1], 0.1846, atol=5e-5) and np.isclose(p_np[4], 0.5737, atol=5e-5)
print("\nSame numbers as Week 3: 0.1846 for the malignant 2 cm tumour, 0.5737 for the")
print("benign 5 cm one. A single sigmoid neuron is logistic regression -- nothing more.")

`Input(shape=(1,))` declares the number of features up front, so the weights are created
immediately rather than on the first call. `Sequential` wraps the layer in a model, which gives
you `predict`. `predict` always returns a NumPy array, not a tensor.

---

## 5. A layer of three neurons

A layer is several neurons that all receive the *same* input. TensorFlow stores their weights
together in one matrix $\mathbf{W}$ of shape `(n_in, units)`: **column $j$ is the weight vector
of unit $j$**. The biases form a vector of length `units`.

These are the layer-1 parameters from the lecture's worked example:

$$\mathbf{w}^{[1]}_1 = \begin{bmatrix}1\\2\end{bmatrix},\;
  \mathbf{w}^{[1]}_2 = \begin{bmatrix}-3\\4\end{bmatrix},\;
  \mathbf{w}^{[1]}_3 = \begin{bmatrix}5\\-6\end{bmatrix},\qquad
  \mathbf{b}^{[1]} = [-1,\ 1,\ 2]$$

In [ ]:
W1 = np.array([[1, -3,  5],
               [2,  4, -6]], dtype=float)     # columns are units
b1 = np.array([-1, 1, 2], dtype=float)

layer_1 = Dense(units=3, activation="sigmoid")
_ = layer_1(np.zeros((1, 2)))                  # build it: n_in = 2
assert layer_1.get_weights()[0].shape == W1.shape
layer_1.set_weights([W1, b1])

x = np.array([[1.0, 2.0]])                     # one example: shape (1, 2)
a1 = layer_1(x).numpy()
print("a1 =", a1, "  shape", a1.shape)

# The board, line by line.
z_hand = np.array([1*1 + 2*2 - 1, -3*1 + 4*2 + 1, 5*1 - 6*2 + 2])
print("z  =", z_hand)
assert np.array_equal(z_hand, [4, 6, -5])
assert np.allclose(a1[0], sigmoid(z_hand), atol=1e-6)
assert np.allclose(a1[0], [0.9820, 0.9975, 0.0067], atol=5e-5)
print("\nMatches the lecture table: 0.9820, 0.9975, 0.0067.")

> **Rows or columns?** Some texts, and the lecture's notation $\mathbf{w}^{[l]}_j$, think of
> each unit's weights as a vector. TensorFlow stacks those vectors as **columns** of
> $\mathbf{W}$, so that a batch of examples $\mathbf{X}$ of shape `(m, n_in)` can be pushed
> through the layer as $\mathbf{X}\mathbf{W}$, of shape `(m, units)`. Pass the transpose by
> mistake and, for a square $\mathbf{W}$, nothing will complain.

In [ ]:
# What the transpose mistake looks like when W is not square: TensorFlow refuses.
try:
    layer_1.set_weights([W1.T, b1])
except ValueError as exc:
    print("ValueError:", str(exc).splitlines()[0][:110])

# And when it is square, nothing refuses: a (2, 2) layer accepts W and W.T equally happily.
sq = Dense(units=2, activation="sigmoid")
_ = sq(np.zeros((1, 2)))
M = np.array([[1.0, 0.0], [5.0, 1.0]])
sq.set_weights([M, np.zeros(2)])
out_M = sq(x).numpy()
sq.set_weights([M.T, np.zeros(2)])
out_MT = sq(x).numpy()
print("\nsquare W  :", out_M, "\nsquare W.T:", out_MT)
assert not np.allclose(out_M, out_MT)
print("Different answers, no error. Shape checks catch only some bugs.")

---

## 6. Two layers: the lecture's network, end to end

The output layer is one sigmoid unit reading the three activations of layer 1, with
$\mathbf{w}^{[2]}_1 = [2, -1, -3]$ and $b^{[2]}_1 = 0.5$. `Sequential` strings the two layers
together, so `predict` runs the whole forward propagation.

In [ ]:
W2 = np.array([[2], [-1], [-3]], dtype=float)  # (n_in = 3, units = 1)
b2 = np.array([0.5])

net = Sequential([
    Input(shape=(2,)),
    Dense(units=3, activation="sigmoid", name="layer1"),
    Dense(units=1, activation="sigmoid", name="layer2"),
])
net.get_layer("layer1").set_weights([W1, b1])
net.get_layer("layer2").set_weights([W2, b2])

X_demo = np.array([[1.0, 2.0],
                   [-1.0, -1.0]])
a2 = net.predict(X_demo, verbose=0)
print("a2 =", a2.ravel())

assert a2.shape == (2, 1)
assert np.allclose(a2.ravel(), [0.8094, 0.0562], atol=5e-5)
yhat = (a2 >= 0.5).astype(int).ravel()
print("yhat =", yhat)
assert np.array_equal(yhat, [1, 0])
print("\nBoth examples match the lecture: 0.8094 -> 1 and 0.0562 -> 0.")

`model.summary()` lists the layers, their output shapes and their parameter counts. `None` in an
output shape stands for "any number of examples": the batch size is not part of the model.

In [ ]:
net.summary()

# One weight per (input, unit) pair, plus one bias per unit.
def dense_params(n_in, units):
    return n_in * units + units

counts = [layer.count_params() for layer in net.layers]
print("per layer:", counts, "  total:", net.count_params())
assert counts == [dense_params(2, 3), dense_params(3, 1)] == [9, 4]
assert net.count_params() == 13

### Checkpoint 1

The lecture's handwritten-digit network has layer sizes $64 \to 25 \to 15 \to 1$, all sigmoid.

1. Build it with `Sequential` (start with `Input(shape=(64,))`) and print its `summary()`.
2. Confirm, with `assert`, that each layer's `count_params()` equals `dense_params(n_in, units)`.
3. Without running anything, predict the total for $64 \to 32 \to 16 \to 1$, then check it.

*Expected:* `1625`, `390` and `16` parameters per layer, `2031` in total; the second network
has `2625` in total.

In [ ]:
# Your code here

### Checkpoint 2

Use `net` from section 6 to classify the example $\mathbf{x} = [0.5,\ -0.5]$. Then compute the
same number by hand with NumPy &mdash; $\mathbf{z}^{[1]}$, $\mathbf{a}^{[1]}$, $z^{[2]}$,
$a^{[2]}$, one line each &mdash; and assert that the two agree. Watch the input's shape.

*Expected:* $\mathbf{z}^{[1]} = [-1.5,\ -2.5,\ 7.5]$, $a^{[2]} \approx$ `0.0989`, so the
prediction is `0`.

In [ ]:
# Your code here

---

## 7. Recap

- TensorFlow wants data as a 2-D matrix, shape `(m, n)`: one row per example. A single example is
  `(1, n)`; use `reshape(1, -1)` to make one.
- Layers return `tf.Tensor`s; `.numpy()` converts back. `predict` returns NumPy directly.
  TensorFlow computes in float32, so compare with a tolerance.
- `Dense(1, activation="linear")` is linear regression; `Dense(1, activation="sigmoid")` is
  logistic regression. Loaded with Week 1 and Week 3 parameters, they reproduce those weeks'
  numbers.
- A layer's weights are a matrix of shape `(n_in, units)` &mdash; **column $j$ is unit $j$**
  &mdash; and a bias vector of length `units`. They do not exist until the input size is known.
- A `Dense` layer has $n_{\text{in}} \cdot \text{units} + \text{units}$ parameters.
- The lecture's $2 \to 3 \to 1$ network gives `0.8094` for $[1, 2]$ and `0.0562` for $[-1, -1]$,
  in TensorFlow exactly as on the board.

### Exercises to hand in

1. Build a single sigmoid neuron with **two** inputs and set its weights to the Week 3 loans
   model of your choice. Verify its predictions on five borrowers against NumPy.
2. Show, by building both and comparing outputs on random inputs, that a network of two
   `Dense` layers **with linear activations** computes exactly the same function as a single
   linear layer. Give the single layer's $\mathbf{W}$ and $\mathbf{b}$ in terms of the two
   layers' parameters. (This is why the activation function matters &mdash; Week 5.)
3. A network has input size 784 and three hidden layers of 128, 64 and 32 units, then one output
   unit. Compute its parameter count by hand, then confirm it in TensorFlow. Which layer holds
   most of the parameters, and why?
4. Write a function `describe(model)` that prints, for every `Dense` layer, the shapes of
   $\mathbf{W}$ and $\mathbf{b}$, the activation's name, and the parameter count, without
   calling `summary()`.

### Next

**Part 2 &mdash; Coffee roasting in TensorFlow:** a network that has to *learn* its weights,
on data that no single neuron can classify.